In [1]:
import os

import numpy as np
import faiss
import pandas as pd
import sqlite3
import datetime
import torch
from pathlib import Path
from itertools import combinations
import json
from sklearn.metrics import roc_auc_score, roc_curve

ROOT = Path.cwd().parents[1]

EMBED_PATH = ROOT / "data/embeddings/base_embeds.pt"
EMBED_NAME = EMBED_PATH.stem

IMAGE_DIR = ROOT / "images/ellipsoid" / EMBED_NAME
CACHE_DIR = ROOT / "data/cache" / EMBED_NAME
RESULTS_DIR = ROOT / "data/results" / EMBED_NAME

EXPERIMENTS_DIR = ROOT / "data/experiments" / EMBED_NAME


In [2]:
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)

In [3]:
embeds = torch.load(EMBED_PATH, weights_only=False)
cls_tokens = embeds["cls_tokens"]

In [4]:
conn = sqlite3.connect(ROOT / "data/sql/metadata.db")

meta = pd.read_sql_query("SELECT * FROM meta", conn)
categories = pd.read_sql_query("SELECT DISTINCT category FROM meta", conn)["category"].to_list()

conn.close()

In [ ]:
category = "bottle"

train_mask = meta["split"] == "train"
train_meta = meta[train_mask]
test_meta = meta[~train_mask]

train_cat_mask = train_meta["category"] == category

good_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] == "good")
defect_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] != "good")

train_emb = cls_tokens[train_mask]
cat_emb = train_emb[train_cat_mask]

test_emb = cls_tokens[~train_mask]
defect_test_emb = test_emb[defect_test_cat_mask]
good_test_emb = test_emb[good_test_cat_mask]

In [6]:
def fit_ellipsoid(covered_idx, cat_emb, weights, reg=1e-4):
    X = cat_emb[covered_idx]

    w = weights / weights.sum()
    center = np.average(X, axis=0, weights=w)
    diff = X - center

    if len(X) > 1:
        cov = (diff * w[:, None]).T @ diff 

        correction = 1.0 / (1.0 - np.sum(w ** 2))
        cov *= correction
    else:
        cov = np.eye(X.shape[1], X.shape[1])

    # Regularise because local regions may have very few points
    cov += np.eye(cov.shape[0]) * reg

    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    eigvals = np.maximum(eigvals, reg)

    proj = diff @ eigvecs

    d2 = np.sum((proj ** 2) / eigvals, axis=1)
    threshold = (d2 * weights).max()

    eig_ratio = eigvals.max() / eigvals.min() 

    return center, eigvecs, eigvals, threshold, eig_ratio

In [7]:
def fit_supported_ellipsoid(covered_idx, ellipsoids, weights, cat_emb, min_points=5, reg=1e-4):
    X = cat_emb[covered_idx]
    w = weights / weights.sum()
    
    center = X.mean(axis=0)
    diff = X - center

    if len(X) > 1:
        own_cov = (diff * w[:, None]).T @ diff 

        correction = 1.0 / (1.0 - np.sum(w ** 2))
        own_cov *= correction
    else:
        own_cov = np.eye(X.shape[1])

    support_id = np.argmin([
        np.linalg.norm(center - e["center"]) 
        for e in ellipsoids
    ])
    support = ellipsoids[support_id]

    sup_cov = support["eigvecs"] @ np.diag(support["eigvals"]) @ support["eigvecs"].T

    alpha = min(1.0, len(X) / min_points)
    cov = alpha * own_cov + (1 - alpha) * sup_cov
    cov += np.eye(cov.shape[0]) * reg

    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals = np.maximum(eigvals[order], reg)
    eigvecs = eigvecs[:, order]

    diff = X - center
    proj = diff @ eigvecs
    d2 = np.sum((proj ** 2) / eigvals, axis=1)

    threshold = (d2 * weights).max() if len(d2) else 0.0

    # singleton fallback scale
    if threshold == 0.0:
        threshold = support["threshold"] * (1.0 - alpha)

    eig_ratio = eigvals.max() / eigvals.min()

    return center, eigvecs, eigvals, threshold, eig_ratio, support_id
    

In [8]:
def grow_ellipsoid(X, center, eigvecs, eigvals, threshold, growth, min_growth=5e-3):
    diff = X - center
    proj = diff @ eigvecs

    var_ratio = eigvals / eigvals.max()
    axis_growth = 1.0 + (growth - 1) * np.clip(var_ratio, min_growth, 1.0)

    grown_eigvals = eigvals * (axis_growth ** 2)

    d2 = np.sum((proj ** 2) / grown_eigvals, axis=1)
    return d2 <= threshold

In [9]:
def ellipsoid_inside(X, center, eigvecs, eigvals, threshold):
    diff = X - center
    proj = diff @ eigvecs
    d2 = np.sum((proj ** 2) / eigvals, axis=1)
    return d2 <= threshold

In [10]:
def find_weight(weights, worst_local, covered_idx, cat_emb, uncovered_mask, space=10):
    lo = 0
    hi = weights[worst_local]

    best = lo
    test_weights = weights.copy()
    for _ in range(space):
        mid = (lo + hi) / 2
        
        test_weights[worst_local] = mid
        center, eigvecs, eigvals, threshold, _ = fit_ellipsoid(covered_idx, cat_emb, test_weights)

        inside_all = ellipsoid_inside(cat_emb, center, eigvecs, eigvals, threshold)
        shared_idx = np.where((inside_all) & (~uncovered_mask))[0]

        if len(shared_idx) == 0:
            best = mid
            lo = mid
        else:
            hi = mid
    
    weights[worst_local] = best
    return weights

In [11]:
def clean_candidate(covered_idx, cat_emb, uncovered_mask, weights, ellipsoids, support_min_points=5, min_points=1, reg=1e-4):
    """
    Iteratively emoved embeddings form a candidate hypersphere until it statisifes the
    exclusive embedding embedding assignment constraint

    Starting from a candiate set of embeddings, the function repeatedly computed the
    centroid and radius of the hyphersphere. If the resulting hyphersphere contains embeddings that 
    have been assinged to previous hyperspheres, the embedding contributing most to the current radius
    is removed and the hypersphere is recomputed. This process continues until no previously assinged 
    embedings lie within the hypersphere or onbly a single embedding remains

    Parameters
    ----------
    covered_idx : ndarray
        Indices of embeddings current assigned to the candidate hypersphere

    cat_emb : ndarray
        Embedding matrix of the current object category

    uncovered_mask : ndarray
        Boolean mask indicating which embeddings have not yet been assigned
        to a hypersphere

    min_points : int
        The minimum anoubt of points a sphere should keep. Used to ensure previously
        accepted candidates are not rejected.
        
    Returns
    -------
    covered_idx : ndarray
        Indices of the cleaned hypersphere

    centroid : ndarray
        Centroid of the final hypersphere
    
    radius : float
        Radius of the final hypersphere. (Singleton spheres have radius 0)
    """

    ellipse_id = None

    while len(covered_idx) > min_points:
        if len(covered_idx) < support_min_points and len(ellipsoids) > 0:
            center, eigvecs, eigvals, threshold, eig_ratio, ellipse_id = fit_supported_ellipsoid(
                covered_idx, ellipsoids, weights, cat_emb, min_points=support_min_points, reg=reg
                )
        else:
            center, eigvecs, eigvals, threshold, eig_ratio = fit_ellipsoid(
                covered_idx, cat_emb, weights, reg=reg
                )

        inside_all = ellipsoid_inside(cat_emb, center, eigvecs, eigvals, threshold)
        shared_idx = np.where((inside_all) & (~uncovered_mask))[0]

        if len(shared_idx) == 0:
            return covered_idx, center, eigvecs, eigvals, threshold, eig_ratio, weights, ellipse_id

        print("Enroach Detected")
        shared_diff = cat_emb[shared_idx] - center
        shared_proj = shared_diff @ eigvecs
        shared_contrib = (shared_proj ** 2) / eigvals

        bad_shared_local = shared_contrib.sum(axis=1).argmax()
        bad_axis = shared_contrib[bad_shared_local].argmax()

        cand_diff = cat_emb[covered_idx] - center
        cand_proj = cand_diff @ eigvecs
        cand_axis_contrib = (cand_proj[:, bad_axis] ** 2) / eigvals[bad_axis]

        worst_local = cand_axis_contrib.argmax()
        weights = find_weight(weights, worst_local, covered_idx, cat_emb, uncovered_mask)

        if np.isclose(weights[worst_local], 0.0, atol=1e-3):
            covered_idx = np.delete(covered_idx, worst_local)
            weights = np.delete(weights, worst_local)

    if len(covered_idx) < support_min_points:
        center, eigvecs, eigvals, threshold, eig_ratio, ellipse_id = fit_supported_ellipsoid(covered_idx, ellipsoids, weights, cat_emb, min_points=min_points, reg=reg)
    else:
        center, eigvecs, eigvals, threshold, eig_ratio = fit_ellipsoid(covered_idx, cat_emb, weights, reg=reg)
        
    return covered_idx, center, eigvecs, eigvals, threshold, eig_ratio, weights, ellipse_id

In [12]:
time = datetime.datetime.now().strftime("%y-%m-%d_%H-%M-%S")
EXPERIMENTS_CAT_DIR = EXPERIMENTS_DIR / category / time
os.makedirs(EXPERIMENTS_CAT_DIR, exist_ok=True)

uncovered_mask = np.ones(len(cat_emb), dtype=bool)
ellipsoids = []

K_frac = 0.05

start_growth = 1.2
min_growth = 1

min_points = 5

while uncovered_mask.any():
    uncovered_idx = np.where(uncovered_mask)[0]
    uncovered_emb = cat_emb[uncovered_idx].astype("float32")

    K = max(2, int(K_frac * len(uncovered_idx)))
    K = min(K, len(uncovered_idx) - 1)

    growth = max(min_growth, start_growth - 0.0025 * len(ellipsoids))
    
    if K < 1:
        covered_idx = uncovered_idx
        weights = np.ones(len(covered_idx))

        center, eigvecs, eigvals, threshold, eig_ratio, ellipse_id = fit_supported_ellipsoid(covered_idx, ellipsoids, weights, cat_emb, min_points=5, reg=1e-4)

    else:
        index = faiss.IndexFlatL2(uncovered_emb.shape[1])
        index.add(uncovered_emb)

        D, I = index.search(uncovered_emb.astype("float32"), k=K+1) # +1 as nearest is itself 

        D = D[:, 1:]    # Remove self
        I = I[:, 1:]

        avg_knn = D.mean(axis=1)

        local_compact  = avg_knn.argmin()
        neighbours_local = I[local_compact]

        full_covered_idx = uncovered_idx[np.r_[local_compact, neighbours_local]]

        weights = np.ones(len(full_covered_idx))
        covered_idx, center, eigvecs, eigvals, threshold, eig_ratio, weights, ellipse_id = clean_candidate(
            full_covered_idx,
            cat_emb,
            uncovered_mask,
            weights,
            ellipsoids
        )

        while True:
            old_covered_idx = covered_idx.copy()
            
            candidate_mask = grow_ellipsoid(cat_emb, center, eigvecs, eigvals, threshold, growth)
            full_new_covered_idx = np.where(candidate_mask & uncovered_mask)[0]

            grow_weights = np.ones(len(full_new_covered_idx))
            for i, idx in enumerate(full_new_covered_idx):
                if idx in covered_idx:
                    old_pos = np.where(covered_idx == idx)[0][0]
                    grow_weights[i] = weights[old_pos]
            weights = grow_weights

            new_covered_idx, new_center, new_eigvecs, new_eigvals, new_threshold, new_eig_ratio, new_weights, new_ellipse_id = clean_candidate(
                full_new_covered_idx, 
                cat_emb, 
                uncovered_mask, 
                weights,
                ellipsoids,
                min_points=len(old_covered_idx)
                ) 

            if len(new_covered_idx) > len(old_covered_idx):
                covered_idx = new_covered_idx
                center = new_center
                eigvecs = new_eigvecs
                eigvals = new_eigvals
                threshold = new_threshold
                eig_ratio = new_eig_ratio
                weights = new_weights
                ellipse_id = new_ellipse_id
            else:
                break

    ellipsoids.append({
        "center": center,
        "eigvecs": eigvecs,
        "eigvals": eigvals,
        "threshold": threshold,
        "eig_ratio": eig_ratio,
        "covered_idx": covered_idx,
        "weights": weights,
        "ellipse_id": ellipse_id
    })

    uncovered_mask[covered_idx] = False

ellipsoids_df = pd.DataFrame([
    {
        "n_points": len(e["covered_idx"]),
        "threshold": e["threshold"],
        "eig_ratio": e["eig_ratio"],
        "weights": e["weights"]
    }
    for e in ellipsoids
])

ellipsoids_df.to_csv(EXPERIMENTS_CAT_DIR / f"ellipsoids.csv", index=False)
ellipsoids_df

ValueError: attempt to get argmin of an empty sequence

In [ ]:
overlaps = []

for i, j in combinations(range(len(ellipsoids)), 2):

    e1 = ellipsoids[i]
    e2 = ellipsoids[j]

    # Points owned by j inside ellipsoid i
    diff = cat_emb[e2["covered_idx"]] - e1["center"]
    proj = diff @ e1["eigvecs"]
    d2 = np.sum((proj ** 2) / e1["eigvals"], axis=1)
    j_inside_i = np.sum(d2 <= e1["threshold"])

    # Points owned by i inside ellipsoid j
    diff = cat_emb[e1["covered_idx"]] - e2["center"]
    proj = diff @ e2["eigvecs"]
    d2 = np.sum((proj ** 2) / e2["eigvals"], axis=1)
    i_inside_j = np.sum(d2 <= e2["threshold"])

    overlaps.append({
        "ellipsoid_i": i,
        "ellipsoid_j": j,
        "j_points_inside_i": j_inside_i,
        "i_points_inside_j": i_inside_j,
        "overlap": (j_inside_i + i_inside_j) > 0
    })

overlap_df = pd.DataFrame(overlaps)

overlap_df.to_csv(EXPERIMENTS_CAT_DIR / f"overlaps.csv", index=False)
overlap_df

,ellipsoid_i,ellipsoid_j,j_points_inside_i,i_points_inside_j,overlap
0,0,1,0,0,False
1,0,2,0,0,False
2,0,3,0,0,False
3,0,4,0,0,False
4,0,5,0,0,False
...,...,...,...,...,...
1076,43,45,0,0,False
1077,43,46,0,0,False
1078,44,45,0,0,False
1079,44,46,0,0,False


In [ ]:
num_overlap = overlap_df["overlap"].sum()

num_overlap

np.int64(0)

In [ ]:
def inside_any_count(X, ellipsoids):
    inside_any = np.zeros(len(X), dtype=bool)
    inside_count = np.zeros(len(X), dtype=int)

    for e in ellipsoids:
        diff = X - e["center"]
        proj = diff @ e["eigvecs"]
        d2 = np.sum((proj ** 2) / e["eigvals"], axis=1)

        inside = d2 <= e["threshold"]

        inside_any |= inside
        inside_count += inside

    return inside_any, inside_count

In [ ]:
good_any, good_counts = inside_any_count(good_test_emb, ellipsoids)
defect_any, defect_counts = inside_any_count(defect_test_emb, ellipsoids)

print("Good accepted:", good_any.sum(), "/", len(good_test_emb))
print("Defect accepted:", defect_any.sum(), "/", len(defect_test_emb))

print("\nGood multi-ellipsoid:", np.sum(good_counts > 1))
print("Defect multi-ellipsoid:", np.sum(defect_counts > 1))

Good accepted: 0 / 21
Defect accepted: 0 / 57

Good multi-ellipsoid: 0
Defect multi-ellipsoid: 0


In [ ]:
X = np.vstack([good_test_emb, defect_test_emb])

y_true = np.concatenate([
    np.zeros(len(good_test_emb)),
    np.ones(len(defect_test_emb))
])

n_points_grid = np.array([len(e["covered_idx"]) for e in ellipsoids])
eigval_ratio_grid = np.array([e["eig_ratio"] for e in ellipsoids])

margins = np.full((len(X), len(ellipsoids)), np.inf)

for i, e in enumerate(ellipsoids):
    diff = X - e["center"]
    proj = diff @ e["eigvecs"]
    d2 = np.sum((proj ** 2) / e["eigvals"], axis=1)

    # <0 inside, >0 outside
    margins[:, i] = d2 - e["threshold"]

best_ellipsoid = margins.argmin(axis=1)
scores = margins.min(axis=1)

auroc = roc_auc_score(y_true, scores)
print("AUROC:", auroc)

fpr, tpr, thresholds = roc_curve(y_true, scores)
youden_index = tpr - fpr
best_threshold = thresholds[youden_index.argmax()]

predicted_label = (scores > best_threshold).astype(int)  
correct = (predicted_label == y_true)

results_df = pd.DataFrame({
    "y_true": y_true,
    "score": scores,
    "correct": correct,
    "winning_ellipsoid": best_ellipsoid,
    "winning_n_points": n_points_grid[best_ellipsoid],
    "winning_eigval_ratio": eigval_ratio_grid[best_ellipsoid],
})

# bucket by n_points and check error rate per bucket
results_df["n_points_bucket"] = pd.cut(results_df["winning_n_points"], bins=[0, 3, 5, 10, 100])
print(results_df.groupby("n_points_bucket")["correct"].agg(["mean", "count"]))

# same for eigval_ratio
results_df["eigval_ratio_bucket"] = pd.qcut(results_df["winning_eigval_ratio"], q=4, duplicates="drop")
print(results_df.groupby("eigval_ratio_bucket")["correct"].agg(["mean", "count"]))

print(results_df.groupby(["n_points_bucket", "y_true"])["correct"].agg(["mean", "count"]))

for bucket, group in results_df.groupby("eigval_ratio_bucket"):
    if group["y_true"].nunique() < 2:
        print(bucket, " only one class present, skip")
        continue
    bucket_auroc = roc_auc_score(group["y_true"], group["score"])
    print(bucket, "AUROC:", bucket_auroc, "n:", len(group))

results_df.to_csv(EXPERIMENTS_CAT_DIR / f"results.csv", index=False)

AUROC: 0.7084377610693399
                     mean  count
n_points_bucket                 
(0, 3]           0.679487     78
Empty DataFrame
Columns: [mean, count]
Index: []
                            mean  count
n_points_bucket y_true                 
(0, 3]          0.0     0.714286     21
                1.0     0.666667     57


In [ ]:
metadata = {
    "category": category,
    "K_frac": K_frac,
    "start_growth": start_growth,
    "min_growth": min_growth,
    "reg": 1e-4,
    "growth_type": "variance_scaled",
    "cleaner": "shared_axis",
    "n_ellipsoids": len(ellipsoids),
    "auroc": auroc,
    "good_inside": int(good_any.sum()),
    "defect_inside": int(defect_any.sum())
}

with open(EXPERIMENTS_CAT_DIR / f"metadata.json", "w") as f:
    json.dump(metadata, f)


: 